# ENGRAMA V4 vs ablaciones vs Transformer — 2×T4, GPT-2, 100M tokens

Comparación **controlada** de cuatro modelos de ~20M parámetros, entrenados con el **mismo** tokenizer GPT-2, el **mismo** corpus (~100M tokens), las **mismas** épocas y el **mismo** recetario de entrenamiento (DDP + AMP fp16 + CE lineal fusionada).

| Modelo | Qué es | Qué se le quita |
|---|---|---|
| `engrama_v4` | ENGRAMA V4 **completo** | nada (dual gating + Trace Tap T0 + offsets resonantes + RMSNorm + latent fusion) |
| `engrama_source_gate` | ENGRAMA V4 limitado | **gating dual target-source** → gating V3 source-only |
| `engrama_no_tracetap` | ENGRAMA V4 limitado | **Trace Tap T0** (bypass a la huella prístina) |
| `transformer` | Decoder GPT con RoPE + RMSNorm + SDPA causal | — (baseline con atención \(O(N^2)\)) |

**Hardware objetivo:** Kaggle 2× Tesla T4 16 GB. **Presupuesto:** < 3 h para los 4 entrenamientos + mediciones.

- Tokenizer: **GPT-2 BPE** (50,257)
- Contexto de entrenamiento: **512**
- Tokens de train: **100,000,000** (1 época)
- Batch: **16 / GPU** (global 32 en 2×T4)
- Optimizador: AdamW fused `lr=3e-4`, `betas=(0.9, 0.95)`, warmup 500 + cosine, clip 1.0
- Pérdida: proyección + CE **por chunks de posiciones** (nunca se materializa `(B,N,50k)` completo; CE en fp32)
- Multi-GPU: **DDP/NCCL** (no `DataParallel`)

Al final: pérdida, perplejidad, tokens/s, VRAM vs contexto, orden de complejidad, recuperación KV de largo alcance, y una tabla única de comparación.

> Autor: BUEORM · Licencia AGPL-3.0 · Arquitectura ENGRAMA intacta (solo se optimiza el runtime).


## 0. Instalación y flags CUDA (anti-NaN)


In [ ]:
import os, sys, math, time, json, gc, glob, shutil, subprocess, random, traceback
from pathlib import Path

# Prefer a local checkout (this repo) over PyPI; fall back to GitHub.
def _install():
    cands = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/kaggle/working/engrama'),
        Path('/kaggle/input/engrama'),
    ]
    for p in cands:
        if (p / 'src' / 'engrama').is_dir() and (p / 'pyproject.toml').is_file():
            print('Instalando ENGRAMA editable desde', p)
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(p),
                                   'transformers', 'numpy'])
            return
    print('Instalando ENGRAMA desde GitHub')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'git+https://github.com/bueormnew/engrama.git',
                           'transformers', 'numpy'])

_install()

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import engrama
from engrama import EngramaConfig, EngramaModel, Generator, linear_cross_entropy

print('ENGRAMA', engrama.__version__, '| torch', torch.__version__)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '30')
os.environ.setdefault('NCCL_P2P_DISABLE', '1')
os.environ.setdefault('NCCL_IB_DISABLE', '1')
os.environ.setdefault('TORCHINDUCTOR_FX_GRAPH_CACHE', '1')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NGPU = torch.cuda.device_count() if DEVICE == 'cuda' else 0
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True
    if hasattr(torch.backends.cuda.matmul, 'allow_fp16_reduced_precision_reduction'):
        # GEMM fp16 con reduccion fp16 desborda el residual V4 a Inf/NaN tras el warmup.
        torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
    if hasattr(torch.backends.cuda.matmul, 'allow_tf32'):
        torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch, 'set_float32_matmul_precision'):
        torch.set_float32_matmul_precision('high')
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    if hasattr(torch.backends.cuda, 'enable_flash_sdp'):
        torch.backends.cuda.enable_flash_sdp(False)  # T4 = SM75, sin FlashAttention
    print('GPUs:', [torch.cuda.get_device_name(i) for i in range(NGPU)])
    for i in range(NGPU):
        print('  cuda:%d  %.1f GiB' % (i, torch.cuda.get_device_properties(i).total_memory / 2**30))
else:
    print('SIN GPU — activa 2x T4 en Kaggle. FAST_MODE permite un humo en CPU.')


## 1. Configuración central (un solo sitio)


In [ ]:
FAST_MODE = False   # True = humo (pocos pasos). False = experimento real < 3 h en 2x T4.
SEED = 1234
ARCHS = ['engrama_v4', 'engrama_source_gate', 'engrama_no_tracetap', 'transformer']

SEQ_LEN = 512
VOCAB_SIZE = 50257
TARGET_TRAIN_TOKENS = 100_000_000
TARGET_VALID_TOKENS = 2_000_000

# Identicos para los 4 modelos
EPOCHS = 1
LOCAL_BATCH = 16          # por GPU; global = LOCAL_BATCH * nproc
EVAL_BATCH = 8
LR = 3e-4                 # 6e-4 + AMP fp16 reventaba a NaN ~paso 350
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
WARMUP_STEPS = 500
LOG_EVERY = 50
EVAL_EVERY = 500
EVAL_BATCHES = 25
LINEAR_CHUNK = max(2048, LOCAL_BATCH * SEQ_LEN)  # un GEMM de CE, no 4 kernels
COMPILE = True
COMPILE_MODE = 'reduce-overhead'  # CUDA graphs: el modelo 20M está limitado por launches, no por FLOPs
RESUME = True

TRAIN_FILE = 'tinystories_train.txt'
VALID_FILE = 'tinystories_valid.txt'
TRAIN_IDS = 'tinystories_train.ids'
VALID_IDS = 'tinystories_valid.ids'
TRAIN_BYTES = 2_227_753_162
VALID_BYTES = 22_502_601

WORK_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.abspath('./compare_working')
SAVE_ROOT = os.path.join(WORK_DIR, 'compare_ckpts')
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(SAVE_ROOT, exist_ok=True)

NPROC = max(1, min(2, NGPU)) if NGPU else 1
GLOBAL_BATCH = LOCAL_BATCH * NPROC
TOKENS_PER_STEP = GLOBAL_BATCH * SEQ_LEN
# 100M / (32*512) ≈ 6104 pasos. Tras quitar syncs CPU y CUDA graphs, ~0.15–0.20 s/paso.
EST_STEPS = TARGET_TRAIN_TOKENS // TOKENS_PER_STEP

if FAST_MODE:
    TARGET_TRAIN_TOKENS = 50_000
    TARGET_VALID_TOKENS = 10_000
    LOCAL_BATCH = 4
    WARMUP_STEPS = 5
    LOG_EVERY = 2
    EVAL_EVERY = 8
    EVAL_BATCHES = 2
    COMPILE = False
    EPOCHS = 1
    NPROC = 1
    GLOBAL_BATCH = LOCAL_BATCH * NPROC
    TOKENS_PER_STEP = GLOBAL_BATCH * SEQ_LEN
    EST_STEPS = 20

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('modo=%s  GPUs=%d nproc=%d  seq=%d  global_batch=%d  ~pasos/modelo=%d' % (
    'FAST' if FAST_MODE else 'FULL', NGPU, NPROC, SEQ_LEN, GLOBAL_BATCH, EST_STEPS))
print('tokens train objetivo=%s  tokens/paso=%s  presupuesto train ≈ %.1f h (4 modelos, 0.28 s/paso)' % (
    format(TARGET_TRAIN_TOKENS, ','), format(TOKENS_PER_STEP, ','),
    (EST_STEPS * 0.28 * 4) / 3600.0))
print('ckpts ->', SAVE_ROOT)
if not FAST_MODE and NGPU == 0:
    raise RuntimeError('El modo FULL necesita GPU (idealmente 2x T4 en Kaggle).')
if not FAST_MODE and NGPU < 2:
    print('AVISO: se esperaban 2x T4. Se entrenara con nproc=%d (mismo recetario, menos tok/s).' % NPROC)


## 2. TinyStories (descarga robusta) + tokenizer GPT-2


In [ ]:
def iter_stories(path):
    buf = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if line.strip():
                buf.append(line.rstrip('\n'))
            elif buf:
                yield ' '.join(buf).strip()
                buf = []
    if buf:
        yield ' '.join(buf).strip()


def download_verified(url, path, expected_bytes, retries=8):
    import urllib.request
    dest = path if os.path.isabs(path) else os.path.join(WORK_DIR, path)
    done = os.path.getsize(dest) if os.path.exists(dest) else 0
    if done == expected_bytes:
        print('  %s: ya completo (%.0f MB)' % (dest, done / 2**20))
        return dest
    if done > expected_bytes:
        os.remove(dest)
        done = 0
    for attempt in range(1, retries + 1):
        mode = 'ab' if done else 'wb'
        headers = {'Range': 'bytes=%d-' % done} if done else {}
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=120) as resp:
                total = done if getattr(resp, 'status', 200) == 206 else 0
                if getattr(resp, 'status', 200) == 200:
                    mode, total = 'wb', 0
                os.makedirs(os.path.dirname(dest) or '.', exist_ok=True)
                with open(dest, mode) as f:
                    while True:
                        chunk = resp.read(4 * 1024 * 1024)
                        if not chunk:
                            break
                        f.write(chunk)
                        total += len(chunk)
                        if total % (256 * 1024 * 1024) < 4 * 1024 * 1024:
                            print('    %s: %.0f MB ...' % (os.path.basename(dest), total / 2**20))
            done = os.path.getsize(dest)
            if done == expected_bytes:
                print('  %s: OK (%.0f MB)' % (dest, done / 2**20))
                return dest
            print('  incompleto (%d != %d); reintento ...' % (done, expected_bytes))
        except Exception as exc:
            done = os.path.getsize(dest) if os.path.exists(dest) else 0
            print('  %s: %s; reintento %d/%d' % (os.path.basename(dest), type(exc).__name__, attempt, retries))
            time.sleep(min(30.0, 2 ** attempt))
            if attempt >= retries:
                raise
    raise RuntimeError('Descarga fallida: ' + url)


def find_kaggle_file(basename, expected):
    for pat in ('/kaggle/input/*/' + basename, '/kaggle/input/**/' + basename):
        for hit in sorted(glob.glob(pat, recursive=True)):
            if os.path.getsize(hit) == expected:
                return hit
    return None

TRAIN_URL = ('https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/'
             'TinyStoriesV2-GPT4-train.txt')
VALID_URL = ('https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/'
             'TinyStoriesV2-GPT4-valid.txt')

print('Localizando TinyStories ...')
train_path = find_kaggle_file('TinyStoriesV2-GPT4-train.txt', TRAIN_BYTES)
valid_path = find_kaggle_file('TinyStoriesV2-GPT4-valid.txt', VALID_BYTES)
if FAST_MODE:
    if valid_path is None:
        valid_path = download_verified(VALID_URL, VALID_FILE, VALID_BYTES)
    train_path = valid_path
else:
    if train_path is None:
        train_path = download_verified(TRAIN_URL, TRAIN_FILE, TRAIN_BYTES)
    else:
        print('  train montado:', train_path)
    if valid_path is None:
        valid_path = download_verified(VALID_URL, VALID_FILE, VALID_BYTES)
    else:
        print('  valid montado:', valid_path)
print('train:', train_path)
print('valid:', valid_path)


In [ ]:
class GPT2Adapter:
    """Tokenizer GPT-2 con la interfaz que espera engrama.Generator."""
    def __init__(self, hf_tok):
        self.tok = hf_tok
        eot = hf_tok.eos_token_id
        self.SPECIAL_TOKENS = {'<eos>': eot, '<bos>': eot, '<pad>': eot}
        self.vocab_size = len(hf_tok)

    def encode(self, text, add_bos=False, add_eos=False):
        ids = list(self.tok.encode(text, add_special_tokens=False))
        if add_bos:
            ids = [self.SPECIAL_TOKENS['<bos>']] + ids
        if add_eos:
            ids = ids + [self.SPECIAL_TOKENS['<eos>']]
        return ids

    def decode(self, ids, skip_special_tokens=True):
        return self.tok.decode(list(ids), skip_special_tokens=skip_special_tokens)

    def encode_batch(self, texts):
        outs = self.tok(list(texts), add_special_tokens=False)['input_ids']
        eos = self.SPECIAL_TOKENS['<eos>']
        return [list(ids) + [eos] for ids in outs]


from transformers import GPT2TokenizerFast
_hf = GPT2TokenizerFast.from_pretrained('gpt2')
if len(_hf) != 50257:
    raise RuntimeError('GPT-2 vocab inesperado: %d' % len(_hf))
tokenizer = GPT2Adapter(_hf)
assert tokenizer.vocab_size == VOCAB_SIZE
EOS_ID = tokenizer.SPECIAL_TOKENS['<eos>']
print('Tokenizer GPT-2 BPE  vocab =', tokenizer.vocab_size, ' eos =', EOS_ID)


## 3. Tokenización streaming → memmap int32 (corte exacto a 100M)


In [ ]:
class MemmapTokenWriter:
    def __init__(self, out_raw, initial_capacity):
        self.path = out_raw
        self.cap = max(1024, int(initial_capacity))
        self.mm = np.memmap(out_raw, dtype=np.int32, mode='w+', shape=(self.cap,))
        self.pos = 0

    def ensure(self, extra):
        if self.pos + extra <= self.cap:
            return
        new_cap = max(self.cap * 2, self.pos + extra)
        self.mm.flush(); del self.mm
        with open(self.path, 'r+b') as f:
            f.truncate(new_cap * 4)
        self.mm = np.memmap(self.path, dtype=np.int32, mode='r+', shape=(new_cap,))
        self.cap = new_cap

    def extend(self, ids):
        self.ensure(len(ids))
        self.mm[self.pos:self.pos + len(ids)] = np.asarray(ids, dtype=np.int32)
        self.pos += len(ids)

    def finalize(self, max_ids=None):
        n = int(self.pos if max_ids is None else min(self.pos, max_ids))
        window = SEQ_LEN + 1
        n = (n // window) * window
        self.mm.flush(); del self.mm
        with open(self.path, 'r+b') as f:
            f.truncate(n * 4)
        return np.memmap(self.path, dtype=np.int32, mode='r', shape=(n,))


def stories_to_memmap(path, out_raw, tokenizer, max_ids, batch_stories=256):
    if os.path.exists(out_raw) and os.path.getsize(out_raw) >= 4 * max_ids:
        n = os.path.getsize(out_raw) // 4
        window = SEQ_LEN + 1
        n = min(n, max_ids)
        n = (n // window) * window
        mm = np.memmap(out_raw, dtype=np.int32, mode='r', shape=(n,))
        print('  reusando %s (%s tokens)' % (out_raw, format(n, ',')))
        return mm
    writer = MemmapTokenWriter(out_raw, initial_capacity=max_ids + 4096)
    batch, n_stories = [], 0
    for story in iter_stories(path):
        batch.append(story)
        if len(batch) < batch_stories:
            continue
        for ids in tokenizer.encode_batch(batch):
            if ids:
                writer.extend(ids)
        n_stories += len(batch)
        batch = []
        if n_stories % 20000 < batch_stories:
            print('  %7d cuentos | %10d tokens ...' % (n_stories, writer.pos))
        if writer.pos >= max_ids:
            break
    if batch and writer.pos < max_ids:
        for ids in tokenizer.encode_batch(batch):
            if ids:
                writer.extend(ids)
        n_stories += len(batch)
    print('  %7d cuentos | %10d tokens (pre-corte)' % (n_stories, writer.pos))
    mm = writer.finalize(max_ids=max_ids)
    print('  corte final: %s tokens (%s ventanas x %d)' % (
        format(len(mm), ','), format(len(mm) // (SEQ_LEN + 1), ','), SEQ_LEN))
    return mm

train_ids_path = os.path.join(WORK_DIR, TRAIN_IDS)
valid_ids_path = os.path.join(WORK_DIR, VALID_IDS)
t0 = time.time()
print('Tokenizando train hasta', format(TARGET_TRAIN_TOKENS, ','), 'tokens ...')
train_mm = stories_to_memmap(train_path, train_ids_path, tokenizer, TARGET_TRAIN_TOKENS)
print('Tokenizando valid hasta', format(TARGET_VALID_TOKENS, ','), 'tokens ...')
valid_mm = stories_to_memmap(valid_path, valid_ids_path, tokenizer, TARGET_VALID_TOKENS)
print('Tokenizacion en %ds | train=%s | valid=%s' % (
    int(time.time() - t0), format(len(train_mm), ','), format(len(valid_mm), ',')))
if not FAST_MODE and len(train_mm) < 90_000_000:
    raise RuntimeError('Train tokenizado demasiado corto: %d (se esperaban ~100M)' % len(train_mm))


## 4. Worker DDP (se escribe a disco para `torchrun`)


In [ ]:
WORKER_SRC = '#!/usr/bin/env python3\n"""DDP trainer for the 2x T4 ENGRAMA-vs-Transformer comparison notebook.\n\nTrains one architecture replica per process. Architecture equations are\nunchanged; this file only owns execution (DDP, AMP, fused CE, compile).\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nimport sys\nimport time\nfrom pathlib import Path\n\nimport torch\nimport torch.distributed as dist\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset, DistributedSampler\n\nfrom engrama import (\n    EngramaConfig,\n    EngramaModel,\n    LanguageModelLoss,\n    adamw,\n    compile_model,\n    configure_cuda,\n    destroy_distributed,\n    init_distributed,\n    linear_cross_entropy,\n    wrap_ddp,\n)\n\n\n# ---------------------------------------------------------------------------\n# Dataset\n# ---------------------------------------------------------------------------\nclass TokenWindows(Dataset):\n    """Non-overlapping windows over a flat little-endian int32 token file."""\n\n    def __init__(self, path: str, sequence_length: int, max_tokens: int | None = None):\n        n = os.path.getsize(path) // 4\n        if max_tokens is not None:\n            n = min(n, int(max_tokens))\n        self.tokens = torch.from_file(path, shared=False, size=n, dtype=torch.int32)\n        self.sequence_length = sequence_length\n        self.window = sequence_length + 1\n        self.n = len(self.tokens) // self.window\n\n    def __len__(self):\n        return self.n\n\n    def __getitem__(self, index):\n        start = index * self.window\n        values = self.tokens[start : start + self.window].to(torch.int64)\n        return values[:-1], values[1:]\n\n    def __getitems__(self, indices):\n        """Vectorized window gather so DataLoader workers do one index, not B."""\n        starts = torch.as_tensor(indices, dtype=torch.long).unsqueeze(1) * self.window\n        gather = starts + torch.arange(self.window, dtype=torch.long)\n        values = self.tokens[gather.reshape(-1)].to(torch.int64).view(len(indices), self.window)\n        return [(values[i, :-1], values[i, 1:]) for i in range(values.size(0))]\n\n\n# ---------------------------------------------------------------------------\n# Transformer baseline (RoPE GPT, tied embeddings, RMSNorm)\n# ---------------------------------------------------------------------------\nclass RMSNorm(nn.Module):\n    def __init__(self, d: int, eps: float = 1e-6):\n        super().__init__()\n        self.eps = eps\n        self.weight = nn.Parameter(torch.ones(d))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        out_dtype = x.dtype\n        x32 = x.float() if x.dtype in (torch.float16, torch.bfloat16) else x\n        y = x32 * torch.rsqrt(x32.pow(2).mean(dim=-1, keepdim=True) + self.eps)\n        return (y * self.weight.float()).to(out_dtype)\n\n\ndef _rope_cache(head_dim: int, seq_len: int, device, dtype, theta: float = 10000.0):\n    half = head_dim // 2\n    freq = 1.0 / (theta ** (torch.arange(0, half, device=device, dtype=torch.float32) / half))\n    t = torch.arange(seq_len, device=device, dtype=torch.float32)\n    freqs = torch.outer(t, freq)  # (T, d/2)\n    cos = torch.cos(freqs).to(dtype)\n    sin = torch.sin(freqs).to(dtype)\n    return cos, sin\n\n\ndef _apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:\n    # x: (B, H, T, D) with even D; rotate (x1, x2) pairs in the last dim.\n    x1 = x[..., ::2]\n    x2 = x[..., 1::2]\n    cos = cos[: x.size(-2)].unsqueeze(0).unsqueeze(0)\n    sin = sin[: x.size(-2)].unsqueeze(0).unsqueeze(0)\n    rot1 = x1 * cos - x2 * sin\n    rot2 = x1 * sin + x2 * cos\n    out = torch.stack((rot1, rot2), dim=-1)\n    return out.flatten(-2)\n\n\nclass TransformerBlock(nn.Module):\n    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.0):\n        super().__init__()\n        if d_model % n_heads != 0:\n            raise ValueError("d_model must be divisible by n_heads")\n        self.n_heads = n_heads\n        self.head_dim = d_model // n_heads\n        self.ln1 = RMSNorm(d_model)\n        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)\n        self.proj = nn.Linear(d_model, d_model, bias=False)\n        self.ln2 = RMSNorm(d_model)\n        self.fc1 = nn.Linear(d_model, d_ff, bias=False)\n        self.fc2 = nn.Linear(d_ff, d_model, bias=False)\n        self.drop = nn.Dropout(dropout)\n\n    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:\n        b, t, d = x.shape\n        h = self.ln1(x)\n        qkv = self.qkv(h).view(b, t, 3, self.n_heads, self.head_dim)\n        q, k, v = qkv.unbind(dim=2)\n        q = _apply_rope(q.transpose(1, 2), cos, sin)\n        k = _apply_rope(k.transpose(1, 2), cos, sin)\n        v = v.transpose(1, 2)\n        attn = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)\n        attn = attn.transpose(1, 2).contiguous().view(b, t, d)\n        x = x + self.drop(self.proj(attn))\n        h = self.ln2(x)\n        x = x + self.drop(self.fc2(F.gelu(self.fc1(h))))\n        return x\n\n\nclass TransformerLM(nn.Module):\n    """Decoder-only Transformer sized to match ENGRAMA (~20M, GPT-2 vocab)."""\n\n    def __init__(\n        self,\n        vocab_size: int,\n        d_model: int = 256,\n        n_layers: int = 9,\n        n_heads: int = 8,\n        d_ff: int = 1024,\n        max_seq_len: int = 2048,\n        dropout: float = 0.0,\n    ):\n        super().__init__()\n        self.vocab_size = vocab_size\n        self.d_model = d_model\n        self.n_layers = n_layers\n        self.n_heads = n_heads\n        self.head_dim = d_model // n_heads\n        self.max_seq_len = max_seq_len\n        self.tok_emb = nn.Embedding(vocab_size, d_model)\n        self.blocks = nn.ModuleList(\n            [TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]\n        )\n        self.ln_f = RMSNorm(d_model)\n        self.drop = nn.Dropout(dropout)\n        # RoPE tables registered as buffers: created once at init so every\n        # forward reads stable tensors.  (Caching them as plain attributes\n        # created inside ``forward`` breaks CUDA-graph replays under\n        # ``torch.compile(mode="reduce-overhead")``: the cached cos/sin get\n        # allocated inside a graph pool and silently die or crash when the\n        # eval batch shape triggers a second recording.)\n        cos, sin = _rope_cache(self.head_dim, max_seq_len, "cpu", torch.float32)\n        self.register_buffer("rope_cos", cos, persistent=False)\n        self.register_buffer("rope_sin", sin, persistent=False)\n        self._init_weights()\n\n    def _init_weights(self) -> None:\n        for m in self.modules():\n            if isinstance(m, nn.Linear):\n                nn.init.normal_(m.weight, mean=0.0, std=0.02)\n            elif isinstance(m, nn.Embedding):\n                nn.init.normal_(m.weight, mean=0.0, std=0.02)\n\n    def num_parameters(self, only_trainable: bool = False) -> int:\n        if only_trainable:\n            return sum(p.numel() for p in self.parameters() if p.requires_grad)\n        return sum(p.numel() for p in self.parameters())\n\n    def forward_features(self, input_ids: torch.Tensor) -> torch.Tensor:\n        b, t = input_ids.shape\n        if t > self.max_seq_len:\n            raise ValueError(f"sequence length {t} exceeds max_seq_len={self.max_seq_len}")\n        x = self.drop(self.tok_emb(input_ids))\n        # Slices of persistent buffers: no dynamic allocation, no staleness.\n        cos = self.rope_cos[:t].to(x.dtype)\n        sin = self.rope_sin[:t].to(x.dtype)\n        for block in self.blocks:\n            x = block(x, cos, sin)\n        return self.ln_f(x)\n\n    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:\n        return F.linear(self.forward_features(input_ids), self.tok_emb.weight)\n\n    def forward_loss(\n        self,\n        input_ids: torch.Tensor,\n        targets: torch.Tensor,\n        *,\n        linear_chunk_size: int = 2048,\n        checkpoint_chunks: bool = False,\n        ignore_index: int = -100,\n    ) -> torch.Tensor:\n        hidden = self.forward_features(input_ids)\n        return linear_cross_entropy(\n            hidden,\n            self.tok_emb.weight,\n            targets,\n            scale=1.0,\n            chunk_size=linear_chunk_size,\n            ignore_index=ignore_index,\n            checkpoint_chunks=checkpoint_chunks,\n        )\n\n\n# ---------------------------------------------------------------------------\n# Architecture factory\n# ---------------------------------------------------------------------------\nENGRAMA_BASE = dict(\n    d_model=256,\n    d_gate=32,\n    d_ff=1024,\n    num_cells=8,\n    num_encoder_layers=2,\n    num_consolidation_layers=9,\n    num_candidates=4,\n    candidate_aggregation="latent_fusion",\n    synapse_rank=32,\n    version="v4",\n    offset_mode="resonant_multirate",\n    gating_mode="dual",\n    trace_tap=True,\n    norm_type="rmsnorm",\n    tie_embeddings=True,\n    stable_init=True,\n)\n\nARCH_SPECS = {\n    "engrama_v4": {\n        "kind": "engrama",\n        "title": "ENGRAMA V4 completo",\n        "overrides": {},\n    },\n    "engrama_source_gate": {\n        "kind": "engrama",\n        "title": "ENGRAMA sin dual gating (source-only)",\n        "overrides": {"gating_mode": "source"},\n    },\n    "engrama_no_tracetap": {\n        "kind": "engrama",\n        "title": "ENGRAMA sin Trace Tap T0",\n        "overrides": {"trace_tap": False},\n    },\n    "transformer": {\n        "kind": "transformer",\n        "title": "Transformer decoder (RoPE, RMSNorm)",\n        "overrides": {},\n    },\n}\n\n\ndef build_raw_model(arch: str, vocab_size: int, seq_len: int) -> nn.Module:\n    if arch not in ARCH_SPECS:\n        raise ValueError(f"unknown arch {arch!r}; choose from {tuple(ARCH_SPECS)}")\n    spec = ARCH_SPECS[arch]\n    if spec["kind"] == "engrama":\n        kw = dict(ENGRAMA_BASE)\n        kw.update(spec["overrides"])\n        cfg = EngramaConfig(vocab_size=vocab_size, context_length=seq_len, **kw)\n        return EngramaModel(cfg)\n    return TransformerLM(\n        vocab_size=vocab_size,\n        d_model=256,\n        n_layers=9,\n        n_heads=8,\n        d_ff=1024,\n        max_seq_len=max(2048, seq_len),\n    )\n\n\ndef model_card(arch: str, model: nn.Module) -> dict:\n    spec = ARCH_SPECS[arch]\n    card = {\n        "arch": arch,\n        "title": spec["title"],\n        "kind": spec["kind"],\n        "parameters": int(model.num_parameters()),\n        "overrides": spec["overrides"],\n    }\n    if spec["kind"] == "engrama":\n        cfg = model.config\n        card.update(\n            {\n                "version": cfg.version,\n                "d_model": cfg.d_model,\n                "num_cells": cfg.num_cells,\n                "num_encoder_layers": cfg.num_encoder_layers,\n                "num_consolidation_layers": cfg.num_consolidation_layers,\n                "gating_mode": cfg.gating_mode,\n                "trace_tap": bool(cfg.trace_tap),\n                "offset_mode": cfg.offset_mode,\n                "norm_type": cfg.norm_type,\n                "candidate_aggregation": cfg.candidate_aggregation,\n                "receptive_field": cfg.receptive_field(),\n            }\n        )\n    else:\n        card.update(\n            {\n                "d_model": model.d_model,\n                "n_layers": model.n_layers,\n                "n_heads": model.n_heads,\n                "positional": "rope",\n                "attention": "causal_sdpa",\n                "norm_type": "rmsnorm",\n            }\n        )\n    return card\n\n\n# ---------------------------------------------------------------------------\n# CLI / training\n# ---------------------------------------------------------------------------\ndef arguments():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--arch", required=True, choices=tuple(ARCH_SPECS))\n    p.add_argument("--train", required=True)\n    p.add_argument("--valid", required=True)\n    p.add_argument("--output", required=True)\n    p.add_argument("--seq-len", type=int, default=512)\n    p.add_argument("--vocab-size", type=int, default=50257)\n    p.add_argument("--batch-size", type=int, default=16, help="per-GPU batch")\n    p.add_argument("--eval-batch-size", type=int, default=8)\n    p.add_argument("--epochs", type=int, default=1)\n    p.add_argument("--lr", type=float, default=3e-4)\n    p.add_argument("--warmup-steps", type=int, default=500)\n    p.add_argument("--weight-decay", type=float, default=0.01)\n    p.add_argument("--grad-clip", type=float, default=1.0)\n    p.add_argument("--workers", type=int, default=2)\n    p.add_argument("--log-every", type=int, default=50)\n    p.add_argument("--eval-every", type=int, default=500)\n    p.add_argument("--eval-batches", type=int, default=25)\n    p.add_argument("--linear-chunk-size", type=int, default=8192)\n    p.add_argument("--max-train-tokens", type=int, default=None)\n    p.add_argument("--max-valid-tokens", type=int, default=None)\n    p.add_argument("--max-steps", type=int, default=0, help="0 = full epoch(s)")\n    p.add_argument("--seed", type=int, default=1234)\n    p.add_argument("--no-compile", action="store_true")\n    p.add_argument("--compile-mode", default="reduce-overhead",\n                   choices=("default", "reduce-overhead", "max-autotune"))\n    p.add_argument("--resume", action="store_true")\n    p.add_argument("--checkpoint-loss", action="store_true")\n    return p.parse_args()\n\n\ndef planned_total_steps(steps_per_epoch: int, epochs: int, max_steps: int = 0) -> int:\n    """Absolute last step (not ``start_step + remaining``). Resume must not retrain."""\n    planned = int(steps_per_epoch) * int(epochs)\n    if max_steps and int(max_steps) > 0:\n        planned = min(planned, int(max_steps))\n    return planned\n\n\ndef make_grad_scaler(amp: bool):\n    try:\n        return torch.amp.GradScaler(\n            "cuda", enabled=amp, init_scale=2**12, growth_interval=2000\n        )\n    except (TypeError, AttributeError):\n        return torch.cuda.amp.GradScaler(\n            enabled=amp, init_scale=2**12, growth_interval=2000\n        )\n\n\ndef shutdown_loader(loader) -> None:\n    """Join DataLoader workers so torchrun can exit (persistent_workers hang otherwise)."""\n    if loader is None:\n        return\n    iterator = getattr(loader, "_iterator", None)\n    if iterator is not None and hasattr(iterator, "_shutdown_workers"):\n        try:\n            iterator._shutdown_workers()\n        except Exception:\n            pass\n\n\ndef reduce_mean(value: torch.Tensor, world_size: int) -> torch.Tensor:\n    if world_size > 1:\n        dist.all_reduce(value, op=dist.ReduceOp.SUM)\n        value /= world_size\n    return value\n\n\n@torch.inference_mode()\ndef evaluate(raw_model, loader, device, amp, max_batches, world_size):\n    """Eval on the *raw* model (never the DDP/compiled wrapper).\n\n    The compiled ``reduce-overhead`` module owns CUDA graphs recorded for the\n    training batch shape; interleaving eval batches through it both re-records\n    graphs (slow) and risks cudagraph-pool errors that killed the transformer\n    baseline mid-run.  ``raw_model`` shares the exact same parameters, so this\n    evaluates the identical function eagerly and safely.\n    """\n    raw_model.eval()\n    total = torch.zeros((), device=device)\n    count = torch.zeros((), device=device)\n    for i, (x, y) in enumerate(loader):\n        if i >= max_batches:\n            break\n        x = x.to(device, non_blocking=True)\n        y = y.to(device, non_blocking=True)\n        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):\n            loss = raw_model.forward_loss(\n                x, y, linear_chunk_size=8192, checkpoint_chunks=False\n            )\n        loss = float(loss.item())  # read immediately: no stale graph outputs\n        if math.isfinite(loss):\n            total += loss\n            count += 1\n    if world_size > 1:\n        dist.all_reduce(total)\n        dist.all_reduce(count)\n    raw_model.train()\n    return (total / count.clamp_min(1)).item()\n\n\ndef save_payload(raw_model, output: Path, card: dict) -> None:\n    output.mkdir(parents=True, exist_ok=True)\n    torch.save(raw_model.state_dict(), output / "model.pt")\n    if hasattr(raw_model, "config"):\n        raw_model.config.save(str(output / "config.json"))\n    else:\n        with open(output / "config.json", "w", encoding="utf-8") as f:\n            json.dump(card, f, indent=2)\n    with open(output / "arch.json", "w", encoding="utf-8") as f:\n        json.dump(card, f, indent=2)\n\n\ndef main():\n    args = arguments()\n    # Kaggle 2x T4 often has flaky P2P/IB; disable to keep NCCL stable.\n    os.environ.setdefault("NCCL_P2P_DISABLE", "1")\n    os.environ.setdefault("NCCL_IB_DISABLE", "1")\n    ctx = init_distributed()\n    configure_cuda()\n    if torch.cuda.is_available():\n        mm = torch.backends.cuda.matmul\n        if hasattr(mm, "allow_fp16_reduced_precision_reduction"):\n            mm.allow_fp16_reduced_precision_reduction = False\n        torch.backends.cuda.enable_mem_efficient_sdp(True)\n        if hasattr(torch.backends.cuda, "enable_flash_sdp"):\n            # T4 (SM75) has no FlashAttention; keep mem-efficient + math.\n            torch.backends.cuda.enable_flash_sdp(False)\n\n    device = (\n        torch.device("cuda", ctx.local_rank)\n        if torch.cuda.is_available()\n        else torch.device("cpu")\n    )\n    amp = device.type == "cuda"\n    torch.manual_seed(args.seed + ctx.rank)\n\n    raw_model = build_raw_model(args.arch, args.vocab_size, args.seq_len).to(device)\n    card = model_card(args.arch, raw_model)\n    optimizer = adamw(\n        raw_model.parameters(),\n        lr=args.lr,\n        weight_decay=args.weight_decay,\n        betas=(0.9, 0.95),\n        fused=amp,\n    )\n    loss_model = LanguageModelLoss(\n        raw_model,\n        linear_chunk_size=args.linear_chunk_size,\n        checkpoint_chunks=args.checkpoint_loss,\n        use_fused_linear_loss=True,\n    )\n    compiled = False\n    compile_mode = args.compile_mode\n    if not args.no_compile and amp:\n        os.environ.setdefault("TORCHINDUCTOR_FX_GRAPH_CACHE", "1")\n        try:\n            loss_model = compile_model(loss_model, enabled=True, mode=compile_mode)\n            compiled = True\n        except Exception as exc:\n            if compile_mode != "default":\n                if ctx.is_main:\n                    print(\n                        f"[compile] {compile_mode} failed ({type(exc).__name__}); "\n                        "falling back to default",\n                        flush=True,\n                    )\n                try:\n                    loss_model = compile_model(loss_model, enabled=True, mode="default")\n                    compiled = True\n                    compile_mode = "default"\n                except Exception as exc2:\n                    if ctx.is_main:\n                        print(f"[compile] disabled ({type(exc2).__name__}: {exc2})", flush=True)\n                    compiled = False\n            else:\n                if ctx.is_main:\n                    print(f"[compile] disabled ({type(exc).__name__}: {exc})", flush=True)\n                compiled = False\n    train_model = wrap_ddp(loss_model, ctx, static_graph=True)\n\n    train_ds = TokenWindows(args.train, args.seq_len, args.max_train_tokens)\n    valid_ds = TokenWindows(args.valid, args.seq_len, args.max_valid_tokens)\n    train_sampler = (\n        DistributedSampler(\n            train_ds,\n            num_replicas=ctx.world_size,\n            rank=ctx.rank,\n            shuffle=True,\n            seed=args.seed,\n            drop_last=True,\n        )\n        if ctx.distributed\n        else None\n    )\n    valid_sampler = (\n        DistributedSampler(\n            valid_ds,\n            num_replicas=ctx.world_size,\n            rank=ctx.rank,\n            shuffle=False,\n        )\n        if ctx.distributed\n        else None\n    )\n    loader_kw = dict(num_workers=args.workers, pin_memory=amp)\n    if args.workers > 0:\n        loader_kw.update(persistent_workers=True, prefetch_factor=4)\n    train_loader = DataLoader(\n        train_ds,\n        batch_size=args.batch_size,\n        sampler=train_sampler,\n        shuffle=train_sampler is None,\n        drop_last=True,\n        **loader_kw,\n    )\n    valid_loader = DataLoader(\n        valid_ds,\n        batch_size=args.eval_batch_size,\n        sampler=valid_sampler,\n        shuffle=False,\n        drop_last=False,\n        **loader_kw,\n    )\n\n    output = Path(args.output)\n    if ctx.is_main:\n        output.mkdir(parents=True, exist_ok=True)\n        save_payload(raw_model, output, card)\n\n    if ctx.distributed:\n        dist.barrier()\n\n    scaler = make_grad_scaler(amp)\n    start_step, best = 0, float("inf")\n    state_path = output / "trainer_state.pt"\n    if args.resume and state_path.exists():\n        checkpoint = torch.load(state_path, map_location=device)\n        raw_model.load_state_dict(checkpoint["model"])\n        optimizer.load_state_dict(checkpoint["optimizer"])\n        scaler.load_state_dict(checkpoint["scaler"])\n        start_step, best = int(checkpoint["step"]), float(checkpoint["best"])\n\n    steps_per_epoch = len(train_loader)\n    total_steps = planned_total_steps(steps_per_epoch, args.epochs, args.max_steps)\n    tokens_per_step = args.batch_size * ctx.world_size * args.seq_len\n\n    if ctx.is_main:\n        print(\n            f"arch={args.arch} params={card[\'parameters\']:,} DDP={ctx.world_size} "\n            f"local_batch={args.batch_size} global_batch={args.batch_size * ctx.world_size} "\n            f"seq={args.seq_len} steps={total_steps} start_step={start_step} "\n            f"compile={compiled}/{compile_mode} amp={amp} ce_chunk={args.linear_chunk_size}",\n            flush=True,\n        )\n        print(f"train_windows={len(train_ds):,} valid_windows={len(valid_ds):,}", flush=True)\n        if start_step >= total_steps:\n            print(\n                f"already finished {args.arch} at step {start_step}/{total_steps}; "\n                "writing metrics and exiting",\n                flush=True,\n            )\n\n    def learning_rate(step: int) -> float:\n        if step < args.warmup_steps:\n            return args.lr * (step + 1) / max(1, args.warmup_steps)\n        progress = (step - args.warmup_steps) / max(1, total_steps - args.warmup_steps)\n        return args.lr * 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))\n\n    history = []\n    skipped = 0\n    step, started = start_step, time.perf_counter()\n    tokens_seen = 0\n    steady_started = None\n    peak_gb = 0.0\n    last_loss = float("nan")\n    last_val = float("nan")\n    mark_cudagraph = hasattr(torch, "compiler") and hasattr(\n        torch.compiler, "cudagraph_mark_step_begin"\n    )\n    train_model.train()\n\n    if amp:\n        torch.cuda.reset_peak_memory_stats(device)\n\n    stop = False\n    if start_step >= total_steps:\n        stop = True\n    for epoch in range(args.epochs):\n        if stop:\n            break\n        if train_sampler is not None:\n            train_sampler.set_epoch(epoch)\n        for x, y in train_loader:\n            if step >= total_steps:\n                stop = True\n                break\n            lr = learning_rate(step)\n            for group in optimizer.param_groups:\n                group["lr"] = lr\n            if mark_cudagraph and compiled:\n                torch.compiler.cudagraph_mark_step_begin()\n            x = x.to(device, non_blocking=True)\n            y = y.to(device, non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):\n                loss = train_model(x, y)\n            # Do not .item()/isfinite the loss every step: that stalls the GPU.\n            # GradScaler already skips the Adam update on inf/NaN grads.\n            scale_before = scaler.get_scale() if amp else 1.0\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            if args.grad_clip > 0:\n                torch.nn.utils.clip_grad_norm_(\n                    raw_model.parameters(), args.grad_clip, foreach=True\n                )\n            scaler.step(optimizer)\n            scaler.update()\n            if amp and scaler.get_scale() < scale_before:\n                skipped += 1\n\n            step += 1\n            tokens_seen += tokens_per_step\n            if step == start_step + 20:\n                steady_started = time.perf_counter()\n\n            if step % args.log_every == 0:\n                logged = reduce_mean(loss.detach(), ctx.world_size).item()\n                last_loss = logged\n                if amp and ctx.is_main:\n                    peak_gb = max(peak_gb, torch.cuda.max_memory_allocated(device) / 2**30)\n                if ctx.is_main:\n                    elapsed = time.perf_counter() - started\n                    done = max(1, step - start_step)\n                    sps = elapsed / done\n                    tps = tokens_per_step / max(1e-9, sps)\n                    history.append({"step": step, "loss": logged, "lr": lr, "tok_s": tps})\n                    extra = f" | skip {skipped}" if skipped else ""\n                    print(\n                        f"step {step:6d}/{total_steps} | loss {logged:.4f} | "\n                        f"lr {lr:.2e} | {sps:.3f}s/step | {tps:,.0f} tok/s{extra}",\n                        flush=True,\n                    )\n\n            if step % args.eval_every == 0 or step == total_steps:\n                if mark_cudagraph and compiled:\n                    # Fresh cudagraph step boundary before switching shapes.\n                    torch.compiler.cudagraph_mark_step_begin()\n                last_val = evaluate(\n                    raw_model,\n                    valid_loader,\n                    device,\n                    amp,\n                    args.eval_batches,\n                    ctx.world_size,\n                )\n                if ctx.is_main:\n                    ppl = (\n                        math.exp(min(20.0, last_val))\n                        if math.isfinite(last_val)\n                        else float("inf")\n                    )\n                    print(\n                        f"  [eval] step {step}: val_loss={last_val:.4f} ppl={ppl:.2f}",\n                        flush=True,\n                    )\n                    if math.isfinite(last_val) and last_val < best:\n                        best = last_val\n                        torch.save(\n                            dict(\n                                model=raw_model.state_dict(),\n                                optimizer=optimizer.state_dict(),\n                                scaler=scaler.state_dict(),\n                                step=step,\n                                best=best,\n                            ),\n                            state_path,\n                        )\n                        torch.save(raw_model.state_dict(), output / "best_model.pt")\n                        save_payload(raw_model, output, card)\n        if stop:\n            break\n\n    elapsed = time.perf_counter() - started\n    done_steps = max(1, step - start_step)\n    overall_tps = (tokens_seen / elapsed) if elapsed > 0 else 0.0\n    if steady_started is not None:\n        steady_elapsed = max(1e-9, time.perf_counter() - steady_started)\n        steady_tokens = max(0, tokens_seen - 20 * tokens_per_step)\n        steady_tps = steady_tokens / steady_elapsed\n    else:\n        steady_tps = overall_tps\n    try:\n        # evaluate() all_reduces: every rank must enter or none must.\n        # Reuse the in-loop eval at total_steps so rank 0 does not wait alone.\n        if not math.isfinite(last_val):\n            last_val = evaluate(\n                raw_model, valid_loader, device, amp,\n                args.eval_batches, ctx.world_size,\n            )\n        val = last_val\n        if ctx.is_main:\n            torch.save(raw_model.state_dict(), output / "model.pt")\n            save_payload(raw_model, output, card)\n            if math.isfinite(val) and val < best:\n                best = val\n            metrics = {\n                "arch": args.arch,\n                "title": card["title"],\n                "kind": card["kind"],\n                "parameters": card["parameters"],\n                "card": card,\n                "seq_len": args.seq_len,\n                "vocab_size": args.vocab_size,\n                "epochs": args.epochs,\n                "steps": step,\n                "planned_steps": total_steps,\n                "global_batch": args.batch_size * ctx.world_size,\n                "tokens_per_step": tokens_per_step,\n                "tokens_seen": tokens_seen,\n                "train_windows": len(train_ds),\n                "valid_windows": len(valid_ds),\n                "final_train_loss": last_loss,\n                "best_val_loss": best if math.isfinite(best) else None,\n                "best_val_ppl": (\n                    math.exp(min(20.0, best)) if math.isfinite(best) else None\n                ),\n                "last_val_loss": val if math.isfinite(val) else None,\n                "seconds": elapsed,\n                "minutes": elapsed / 60.0,\n                "sec_per_step": elapsed / done_steps,\n                "tokens_per_sec_overall": overall_tps,\n                "tokens_per_sec_steady": steady_tps,\n                "peak_train_vram_gb": peak_gb,\n                "skipped_nonfinite": skipped,\n                "world_size": ctx.world_size,\n                "compiled": compiled,\n                "compile_mode": compile_mode,\n                "lr": args.lr,\n                "warmup_steps": args.warmup_steps,\n                "history": history,\n            }\n            with open(output / "metrics.json", "w", encoding="utf-8") as f:\n                json.dump(metrics, f, indent=2)\n            print(\n                f"done {args.arch} in {elapsed/60:.1f} min | best_val={best:.4f} | "\n                f"steady {steady_tps:,.0f} tok/s | skip={skipped}",\n                flush=True,\n            )\n            sys.stdout.flush()\n        if ctx.distributed:\n            dist.barrier()\n    finally:\n        shutdown_loader(train_loader)\n        shutdown_loader(valid_loader)\n        destroy_distributed()\n\n\nif __name__ == "__main__":\n    main()\n'

worker_path = os.path.join(WORK_DIR, 'train_compare_ddp.py')
# Prefer the repo copy when present (keeps notebook and file in sync during development).
for c in (Path.cwd() / 'kaggle' / 'train_compare_ddp.py',
          Path.cwd() / 'train_compare_ddp.py',
          Path(WORK_DIR) / 'train_compare_ddp.py'):
    if c.is_file() and c.stat().st_size > 1000:
        shutil.copy2(c, worker_path)
        print('Worker copiado desde', c)
        break
else:
    with open(worker_path, 'w', encoding='utf-8') as f:
        f.write(WORKER_SRC)
    print('Worker escrito (bundle del notebook) ->', worker_path)

sys.path.insert(0, WORK_DIR)
import importlib
import train_compare_ddp as tcd
importlib.reload(tcd)
print('archs:', list(tcd.ARCH_SPECS))


## 5. Verificar parámetros (~20M, diferencia < 10%)


In [ ]:
cards = {}
for arch in ARCHS:
    m = tcd.build_raw_model(arch, VOCAB_SIZE, SEQ_LEN)
    card = tcd.model_card(arch, m)
    cards[arch] = card
    print('%s: %s params (%.2fM)  |  %s' % (
        arch, format(card['parameters'], ','), card['parameters'] / 1e6, card['title']))
    if arch.startswith('engrama'):
        rf = card['receptive_field']
        print('    gating=%s tap=%s offsets=%s  reach=%s covers=%s' % (
            card['gating_mode'], card['trace_tap'], card['offset_mode'],
            rf['max_reach'], rf['covers_context']))
    del m
    gc.collect()

nparams = [cards[a]['parameters'] for a in ARCHS]
lo, hi, mean = min(nparams), max(nparams), sum(nparams) / len(nparams)
print('\nrango %.2fM — %.2fM  (spread %.1f%% respecto a la media)' % (
    lo / 1e6, hi / 1e6, 100.0 * (hi - lo) / mean))
if lo < 10_000_000 or hi > 22_000_000:
    raise RuntimeError('Parametros fuera de 10-20M: %s' % nparams)
if (hi - lo) / mean > 0.12:
    raise RuntimeError('Los modelos no son comparables en tamaño (spread > 12%).')
print('OK: tamaños comparables.')


## 6. Humo (2–3 pasos) — si esto falla, no se lanza el entrenamiento de 3 h

Un forward+backward corto **por arquitectura** en 1 GPU, batch pequeño, sin `torch.compile`. Detecta OOM, shapes y NaNs inmediatos.


In [ ]:
def smoke_arch(arch, steps=2):
    device = torch.device('cuda:0' if NGPU else 'cpu')
    model = tcd.build_raw_model(arch, VOCAB_SIZE, SEQ_LEN).to(device)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.95))
    B = 2 if device.type == 'cuda' else 1
    T = min(SEQ_LEN, 64 if FAST_MODE else SEQ_LEN)
    x = torch.randint(0, VOCAB_SIZE, (B, T), device=device)
    y = torch.randint(0, VOCAB_SIZE, (B, T), device=device)
    amp = device.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=amp, init_scale=2**12)
    last = None
    for i in range(steps):
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
            loss = model.forward_loss(x, y, linear_chunk_size=1024, checkpoint_chunks=False)
        if not torch.isfinite(loss):
            raise RuntimeError('%s smoke: loss no finita' % arch)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        last = float(loss.detach())
    if device.type == 'cuda':
        torch.cuda.synchronize()
        vram = torch.cuda.max_memory_allocated(device) / 2**30
    else:
        vram = 0.0
    del model, opt, scaler, x, y
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return last, vram

print('Humo ...')
for arch in ARCHS:
    t1 = time.time()
    loss, vram = smoke_arch(arch, steps=2 if not FAST_MODE else 1)
    print('  %-22s  loss=%.4f  vram=%.2f GiB  (%.1fs)' % (arch, loss, vram, time.time() - t1))
print('Humo OK.')


## 7. Entrenamiento DDP — los 4 modelos, uno detrás de otro

Cada `torchrun` usa **las dos T4**, el mismo `.ids`, el mismo batch global, LR, warmup y época.

Entre modelos se libera VRAM. Si un job ya dejó `metrics.json` y `RESUME=True`, **se salta** (útil si Kaggle corta la sesión o si el primer modelo ya terminó).

> Si un run anterior se quedó colgado **después** del último `[eval]` (sin imprimir `done …`), era un deadlock NCCL: el rank 0 volvía a evaluar solo. Re-ejecuta **esta celda** (el worker se reescribe). Con `RESUME=True` no reentrena `engrama_v4`; escribe `metrics.json` y pasa al segundo modelo.


In [ ]:
def _metrics_complete(path):
    if not os.path.isfile(path):
        return None
    try:
        with open(path, 'r', encoding='utf-8') as f:
            metrics = json.load(f)
    except Exception:
        return None
    steps = int(metrics.get('steps') or 0)
    planned = int(metrics.get('planned_steps') or 0)
    if metrics.get('best_val_loss') is None:
        return None
    if planned and steps < planned:
        return None
    return metrics


def _run_tee(cmd, cwd, env, log_path, tail_lines=25):
    # Ejecuta streameando la salida y guardandola completa a disco; si falla,
    # quedan las ultimas lineas impresas y el log completo para post-mortem.
    import subprocess as _sp
    proc = _sp.Popen(cmd, cwd=cwd, env=env, stdout=_sp.PIPE, stderr=_sp.STDOUT,
                     text=True, bufsize=1)
    with open(log_path, 'w', encoding='utf-8') as log:
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
    proc.wait()
    if proc.returncode != 0:
        with open(log_path, 'r', encoding='utf-8') as f:
            tail = f.readlines()[-tail_lines:]
        print('--- ultimas %d lineas de %s ---' % (tail_lines, log_path), flush=True)
        for line in tail:
            print('   ', line.rstrip(), flush=True)
    return proc


def run_train(arch, max_steps=0):
    out = os.path.join(SAVE_ROOT, arch)
    os.makedirs(out, exist_ok=True)
    metrics_path = os.path.join(out, 'metrics.json')
    if RESUME:
        cached = _metrics_complete(metrics_path)
        if cached is not None:
            print('SKIP %s: ya entrenado  step %s/%s  val=%.4f  ppl=%.2f' % (
                arch, cached.get('steps'), cached.get('planned_steps'),
                cached.get('best_val_loss') or float('nan'),
                cached.get('best_val_ppl') or float('nan'),
            ))
            return cached
    cmd = [
        sys.executable, '-m', 'torch.distributed.run',
        '--standalone',
        '--max_restarts', '0',
        '--nproc_per_node', str(NPROC),
        worker_path,
        '--arch', arch,
        '--train', train_ids_path,
        '--valid', valid_ids_path,
        '--output', out,
        '--seq-len', str(SEQ_LEN),
        '--vocab-size', str(VOCAB_SIZE),
        '--batch-size', str(LOCAL_BATCH),
        '--eval-batch-size', str(EVAL_BATCH),
        '--epochs', str(EPOCHS),
        '--lr', str(LR),
        '--warmup-steps', str(WARMUP_STEPS),
        '--weight-decay', str(WEIGHT_DECAY),
        '--grad-clip', str(GRAD_CLIP),
        '--log-every', str(LOG_EVERY),
        '--eval-every', str(EVAL_EVERY),
        '--eval-batches', str(EVAL_BATCHES),
        '--linear-chunk-size', str(LINEAR_CHUNK),
        '--max-train-tokens', str(len(train_mm)),
        '--max-valid-tokens', str(len(valid_mm)),
        '--seed', str(SEED),
        '--compile-mode', COMPILE_MODE,
        '--workers', '2' if DEVICE == 'cuda' and not FAST_MODE else '0',
    ]
    if max_steps:
        cmd += ['--max-steps', str(max_steps)]
    if not COMPILE:
        cmd.append('--no-compile')
    if RESUME:
        cmd.append('--resume')
    print('\n>>>', ' '.join(cmd), flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['TOKENIZERS_PARALLELISM'] = 'false'
    env.setdefault('NCCL_P2P_DISABLE', '1')
    env.setdefault('NCCL_IB_DISABLE', '1')
    env.setdefault('TORCH_NCCL_ASYNC_ERROR_HANDLING', '1')
    env.setdefault('TORCHINDUCTOR_FX_GRAPH_CACHE', '1')
    env['TORCHINDUCTOR_CACHE_DIR'] = os.path.join(WORK_DIR, '.inductor_cache')
    t0 = time.time()
    log_path = os.path.join(SAVE_ROOT, '%s_train.log' % arch)
    proc = _run_tee(cmd, cwd=WORK_DIR, env=env, log_path=log_path)
    elapsed = time.time() - t0
    if proc.returncode != 0:
        print('FALLO %s  rc=%s  (%.1f min)  log completo: %s' % (
            arch, proc.returncode, elapsed / 60.0, log_path), flush=True)
        # Reintento sin torch.compile: compile/cudagraphs es la primera sospechosa
        # (fue lo que mato al transformer en el run anterior: RoPE cacheado +
        # CUDA graphs). El reintento preserva el presupuesto de entrenamiento.
        cleaned, skip = [], False
        for c in cmd:
            if skip:
                skip = False; continue
            if c == '--compile-mode':
                skip = True; continue
            cleaned.append(c)
        cleaned.insert(cleaned.index(worker_path) + 1, '--no-compile')
        print('REINTENTO sin compile:', ' '.join(cleaned), flush=True)
        t0 = time.time()
        proc = _run_tee(cleaned, cwd=WORK_DIR, env=env,
                        log_path=os.path.join(SAVE_ROOT, '%s_retry.log' % arch))
        elapsed = time.time() - t0
    if proc.returncode != 0:
        print('FALLO definitivo %s  rc=%s  (%.1f min)' % (
            arch, proc.returncode, elapsed / 60.0), flush=True)
        return None
    if not os.path.isfile(metrics_path):
        print('FALLO %s: no hay metrics.json' % arch, flush=True)
        return None
    with open(metrics_path, 'r', encoding='utf-8') as f:
        metrics = json.load(f)
    metrics['wall_seconds'] = elapsed
    print('OK %s  val=%.4f  ppl=%.2f  %.0f tok/s  %.1f min' % (
        arch,
        metrics.get('best_val_loss') or float('nan'),
        metrics.get('best_val_ppl') or float('nan'),
        metrics.get('tokens_per_sec_steady') or 0.0,
        elapsed / 60.0,
    ), flush=True)
    return metrics


# Reescribir el worker en esta celda: un re-run no depende de volver a ejecutar la celda 4.
if 'WORKER_SRC' in globals() and 'planned_total_steps' in WORKER_SRC:
    with open(worker_path, 'w', encoding='utf-8') as f:
        f.write(WORKER_SRC)
    print('Worker anti-hang escrito ->', worker_path, flush=True)

all_metrics = {}
train_started = time.time()
MAX_STEPS = EST_STEPS if FAST_MODE else 0
for arch in ARCHS:
    print('\n' + '=' * 72, flush=True)
    print('ENTRENANDO', arch, cards[arch]['title'], flush=True)
    print('=' * 72, flush=True)
    try:
        met = run_train(arch, max_steps=MAX_STEPS)
        all_metrics[arch] = met
    except Exception:
        traceback.print_exc()
        all_metrics[arch] = None
    print('Siguiente modelo...' if arch != ARCHS[-1] else 'Entrenamientos terminados.', flush=True)
    gc.collect()
    if NGPU:
        torch.cuda.empty_cache()

print('\nEntrenamiento total: %.1f min' % ((time.time() - train_started) / 60.0), flush=True)
with open(os.path.join(SAVE_ROOT, 'all_metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(all_metrics, f, indent=2)


## 8. Cargar checkpoints y helpers de inferencia

A partir de aquí todo corre en **1 GPU** (mediciones comparables, sin DDP).


In [ ]:
BENCH_DEVICE = torch.device('cuda:0' if NGPU else 'cpu')

def load_trained(arch):
    model = tcd.build_raw_model(arch, VOCAB_SIZE, SEQ_LEN)
    out = Path(SAVE_ROOT) / arch
    weights = out / 'best_model.pt'
    if not weights.is_file():
        weights = out / 'model.pt'
    if not weights.is_file():
        raise FileNotFoundError('No hay pesos para %s en %s' % (arch, out))
    sd = torch.load(str(weights), map_location='cpu')
    model.load_state_dict(sd)
    model.to(BENCH_DEVICE)
    model.eval()
    return model

loaded = {}
for arch in ARCHS:
    if not all_metrics.get(arch):
        print('skip load', arch, '(sin metricas)')
        continue
    loaded[arch] = load_trained(arch)
    print('cargado', arch, 'params', loaded[arch].num_parameters())


## 9. Tokens/s, VRAM vs contexto, orden de complejidad

Se mide el **forward paralelo** (prefill / paso de entrenamiento a batch 1) a longitudes 64, 128, 256, 512 (y 768 si cabe).

Ajuste \(\log t = a + b \log N\):
- \(b pprox 1\) → **O(N)** (ENGRAMA)
- \(b pprox 2\) → **O(N²)** (atención)
- \(b pprox 0\) → **O(1)**

También se mide **decode incremental** (ENGRAMA `step_forward` con caché jerárquica vs Transformer re-prefill) para ver si el coste por token nuevo es constante.


In [ ]:
# 64..2048: con N<=512 el coste fijo (pesos, launches) domina y toda
# pendiente parece 0; ampliando el rango la pendiente log-log se vuelve honesta.
LENGTHS = [64, 128, 256, 512, 1024, 2048] if not FAST_MODE else [32, 64, 128]


def _sync():
    if BENCH_DEVICE.type == 'cuda':
        torch.cuda.synchronize(BENCH_DEVICE)


@torch.no_grad()
def time_forward(model, seq_len, warmup=3, runs=8):
    x = torch.randint(0, VOCAB_SIZE, (1, seq_len), device=BENCH_DEVICE)
    amp = BENCH_DEVICE.type == 'cuda'
    for _ in range(warmup):
        with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
            _ = model(x)
    _sync()
    if BENCH_DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(BENCH_DEVICE)
    t0 = time.perf_counter()
    for _ in range(runs):
        with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
            _ = model(x)
    _sync()
    dt = (time.perf_counter() - t0) / runs
    peak = (torch.cuda.max_memory_allocated(BENCH_DEVICE) / 2**30) if BENCH_DEVICE.type == 'cuda' else 0.0
    return dt, peak


@torch.no_grad()
def time_engrama_step(model, seq_len, warmup=1):
    # Coste medio de un token incremental con cache jerarquica.
    x = torch.randint(0, VOCAB_SIZE, (1, seq_len), device=BENCH_DEVICE)
    cache = model.get_cache(N_max=seq_len, mode='hierarchical')
    for t in range(seq_len):
        model.step_forward(x[:, t:t+1], cache, timestamp=t)
    # extra token
    tok = x[:, -1:]
    for _ in range(warmup):
        c2 = model.get_cache(N_max=seq_len + 4, mode='hierarchical')
        for t in range(seq_len):
            model.step_forward(x[:, t:t+1], c2, timestamp=t)
    _sync()
    reps = 20
    t0 = time.perf_counter()
    for _ in range(reps):
        model.step_forward(tok, cache, timestamp=seq_len)
    _sync()
    return (time.perf_counter() - t0) / reps


@torch.no_grad()
def time_transformer_reprefill(model, seq_len, warmup=2, runs=6):
    # Decode ingenuo: re-forward de toda la secuencia (O(N^2) por token).
    x = torch.randint(0, VOCAB_SIZE, (1, seq_len), device=BENCH_DEVICE)
    amp = BENCH_DEVICE.type == 'cuda'
    for _ in range(warmup):
        with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
            _ = model(x)
    _sync()
    t0 = time.perf_counter()
    for _ in range(runs):
        with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
            _ = model(x)
    _sync()
    return (time.perf_counter() - t0) / runs


def fit_loglog(ns, ys):
    ns = np.asarray(ns, dtype=np.float64)
    ys = np.asarray(ys, dtype=np.float64)
    mask = np.isfinite(ys) & (ys > 0) & (ns > 0)
    if mask.sum() < 2:
        return float('nan'), 'n/a'
    b, a = np.polyfit(np.log(ns[mask]), np.log(ys[mask]), 1)
    if b < 0.4:
        label = 'O(1) ~ plano'
    elif b < 1.4:
        label = 'O(N)'
    elif b < 1.8:
        label = 'O(N log N) / entre N y N²'
    else:
        label = 'O(N²)'
    return float(b), label


scale_report = {}
for arch, model in loaded.items():
    print('\n--- escala', arch, '---')
    times, mems = [], []
    for n in LENGTHS:
        try:
            dt, peak = time_forward(model, n)
        except Exception as exc:
            print('  N=%d FALLO %s' % (n, type(exc).__name__))
            dt, peak = float('nan'), float('nan')
        times.append(dt)
        mems.append(peak)
        print('  N=%4d  forward=%.4fs  (%.0f tok/s)  peak=%.2f GiB' % (
            n, dt if dt == dt else -1, (n / dt) if dt == dt and dt > 0 else 0, peak if peak == peak else -1))
    b_t, lab_t = fit_loglog(LENGTHS, times)
    b_m, lab_m = fit_loglog(LENGTHS, mems)
    step_times = []
    if arch.startswith('engrama'):
        for n in LENGTHS:
            try:
                step_times.append(time_engrama_step(model, n))
            except Exception as exc:
                print('  step N=%d FALLO %s' % (n, type(exc).__name__))
                step_times.append(float('nan'))
        b_s, lab_s = fit_loglog(LENGTHS, step_times)
        print('  step_forward por token: pendiente=%.2f → %s' % (b_s, lab_s))
        print('  tiempos step ms:', ['%.2f' % (1e3 * t) if t == t else 'nan' for t in step_times])
    else:
        b_s, lab_s = fit_loglog(LENGTHS, times)
        print('  re-prefill por token ≈ forward completo: pendiente=%.2f → %s' % (b_t, lab_t))
        step_times = times
    scale_report[arch] = {
        'lengths': LENGTHS,
        'forward_sec': times,
        'peak_gb': mems,
        'forward_slope': b_t,
        'forward_order': lab_t,
        'memory_slope': b_m,
        'memory_order': lab_m,
        'decode_sec': step_times,
        'decode_slope': b_s,
        'decode_order': lab_s,
    }
    print('  forward pendiente=%.2f → %s | memoria pendiente=%.2f → %s' % (b_t, lab_t, b_m, lab_m))

with open(os.path.join(SAVE_ROOT, 'scale_report.json'), 'w', encoding='utf-8') as f:
    json.dump(scale_report, f, indent=2)


## 9b. Introspección de compuertas (diagnóstico dual vs source)

Mide, con los checkpoints entrenados y un batch real de TinyStories: apertura
media y saturación de \\(\\alpha\\), \\(\\rho\\), \\(\\beta\\) y magnitud del estado por capa.
Si el gating dual satura (\\(\\alpha\\) pegado a 0/1) sus compuertas dejan de
ser modulables y V4 puede perder contra source gating aunque tenga mas
parametros: aqui se ve directamente.


In [ ]:
@torch.no_grad()
def gate_introspection(model, n_tokens=2048):
    x = torch.from_numpy(
        np.asarray(train_mm[:n_tokens], dtype=np.int64)[: (n_tokens // 512) * 512].reshape(-1, 512)
    ).to(BENCH_DEVICE)
    stats = []
    T0 = model.encoder(model.embeddings(x))
    t = T0
    for li, layer in enumerate(model.consolidation.layers):
        mix = layer.mix
        offsets = [p_ for p_ in mix.offsets if p_ < t.size(1)]
        keys = [str(p_) for p_ in offsets]
        k_src = mix._causal_views(mix.p_g_src(t), offsets)
        ws = torch.stack([mix.gate_w_src[k] for k in keys])
        bs = torch.stack([mix.gate_b[k] for k in keys])
        pre = torch.einsum('bnpq,pqd->bnpd', k_src, ws) + bs
        bil_std = 0.0
        if mix.gating_mode == 'dual' and mix.p_g_tgt is not None:
            q = mix.p_g_tgt(t)
            wt = torch.stack([mix.gate_w_tgt[k] for k in keys])
            bil = (q.unsqueeze(2) * k_src).sum(-1, keepdim=True) / math.sqrt(mix.d_gate)
            pre = pre + torch.einsum('bnq,pqd->bnpd', q, wt) + bil
            bil_std = float(bil.std())
        g = torch.sigmoid(pre.float())
        rho = torch.sigmoid(torch.stack([mix.rho[k] for k in keys]).float())
        beta = torch.stack([mix.beta[k] for k in keys]).float().view(-1)
        stats.append({
            'layer': li, 'offsets': offsets,
            'gate_mean': float(g.mean()), 'gate_sat': float(((g > .95) | (g < .05)).float().mean()),
            'bilinear_std': bil_std, 'rho_mean': float(rho.mean()),
            'beta_mean': float(beta.mean()), 'state_std': float(t.float().std()),
        })
        t = layer.forward_train(t, T_0=T0)
    return stats


gates_report = {}
for arch, model in loaded.items():
    if not hasattr(model, 'consolidation'):
        continue
    gates_report[arch] = gate_introspection(model)
    print('---', arch, '---')
    print(' L  offsets            alpha_med  sat5%   bil_std  rho    beta   |T_l|')
    for st in gates_report[arch]:
        print(' %d  %-18s %.3f     %5.1f%%  %.3f    %.3f  %.3f  %.3f' % (
            st['layer'], str(st['offsets']), st['gate_mean'], 100 * st['gate_sat'],
            st['bilinear_std'], st['rho_mean'], st['beta_mean'], st['state_std']))

with open(os.path.join(SAVE_ROOT, 'gates_report.json'), 'w', encoding='utf-8') as f:
    json.dump(gates_report, f, indent=2)


## 10. Recuperación KV — protocolo corregido (3 instrumentos)

El protocolo antiguo usaba como relleno los ids GPT-2 200–250. Esos ids son
**bytes de control** (0x0C–0x1F, DEL, C1: el mapeo byte→id de GPT-2 pone el
espacio en 220 y los bytes de control en 188–254), tokens que **no aparecen
ni una vez** en TinyStories: ningún modelo entrenado en texto natural los ha
visto en contexto, así que el zero-shot era estructuralmente de azar para
cualquier arquitectura (incluido el Transformer).

Tres instrumentos, cada uno midiendo una cosa distinta:

1. **Zero-shot in-distribución**: pares clave-valor con tokens *frecuentes del
   propio train* (elegidos por frecuencia real). Mide si la señal sobrevive sin
   entrenar la tarea. Azar = 1/16 = 6.25 %.
2. **Inducción/copia zero-shot**: una secuencia de 8 tokens frecuentes se
   repite tras un gap; el modelo debe predecir la continuación correcta entre
   las 8 vistas (circuito de inducción). Azar = 12.5 %.
3. **KV entrenado (fine-tune corto, presupuesto idéntico)**: 250 pasos sobre la
   tarea sintética (protocolo adaptado de `benchmarks/kv_retrieval.py` del
   repo). Mide la **capacidad arquitectónica** de vincular y recuperar, sin
   confundirla con el entrenamiento previo en TinyStories.

> Nota teórica: en ENGRAMA el predictor solo ve \(T_L[t]\) (d=256), una
> superposición aditiva de toda la historia con decaimiento por saltos; la
> contribución relativa de un token lejano es ~\(10^{-4}\) del estado
> final. El KV entrenado dice cuánto de ese techo rescatan las compuertas
> duales y el trace tap. (Ver `benchmarks/KV_RETRIEVAL_REPORT.md`: incluso
> entrenado, V3-dyadic quedó en 7.4 % vs 27.5 % de dense_dilated.)


In [ ]:
# ---------- tokens in-distribucion por frecuencia real ----------
import copy as _copy
FREQ_SAMPLE = min(5_000_000, len(train_mm))
counts = np.bincount(np.asarray(train_mm[:FREQ_SAMPLE], dtype=np.int64), minlength=VOCAB_SIZE)
order = [int(t) for t in np.argsort(-counts) if int(t) != EOS_ID]  # ids por frecuencia

def freq_band(lo_rank, hi_rank, n, rng):
    pool = order[lo_rank:hi_rank]
    return rng.sample(pool, n)

KV_SEQ = 192
N_KEYS = 4
N_VALUES = 16
HEADER_SLOTS = [0, 2, 4, 6]
QUERY_POSITIONS = [32, 80, 128, 184]
CHANCE = 1.0 / N_VALUES

_tok_rng = random.Random(4242)
KEY_POOL = freq_band(600, 900, 21, _tok_rng)          # 21 claves frecuentes
VAL_POOL = freq_band(1200, 1600, N_VALUES, _tok_rng)  # 16 valores frecuentes
FILL_POOL = freq_band(100, 400, 12, _tok_rng)         # relleno frecuente
print('claves   :', [repr(tokenizer.decode([t])) for t in KEY_POOL[:5]])
print('valores  :', [repr(tokenizer.decode([t])) for t in VAL_POOL[:5]])
print('relleno  :', [repr(tokenizer.decode([t])) for t in FILL_POOL[:5]])


def make_kv_sample(rng):
    keys = rng.sample(KEY_POOL, N_KEYS)
    values = rng.sample(VAL_POOL, N_KEYS)
    value_of = dict(zip(keys, values))
    seq = [EOS_ID] + [FILL_POOL[0]] * (KV_SEQ - 1)
    used = {0}
    for slot, (k, v) in zip(HEADER_SLOTS, zip(keys, values)):
        seq[1 + slot] = k
        seq[1 + slot + 1] = v
        used.update({1 + slot, 1 + slot + 1})
    query_order = keys[:]
    rng.shuffle(query_order)
    answers = []
    for pos, key in zip(QUERY_POSITIONS, query_order):
        seq[pos] = key
        seq[pos + 1] = value_of[key]
        used.update({pos, pos + 1})
        answers.append((pos, value_of[key]))
    body = [rng.choice(FILL_POOL) for _ in range(8)]
    for i in range(KV_SEQ):
        if i not in used:
            seq[i] = body[i % 8]
    return seq, answers


def logits_at(model, x, positions):
    # Logits de vocabulario SOLO en las posiciones pedidas: (B, K, V)
    feats = model.forward_features(x)
    b, t, d = feats.shape
    sel = (torch.arange(b, device=x.device)[:, None] * t + positions).reshape(-1)
    if hasattr(model, 'evoker'):
        fused = model.evoker.fused_latent(feats)  # latent_fusion / mean
        hf = fused.reshape(-1, d)[sel].view(b, positions.size(1), d)
        logits = F.linear(hf, model.output_embeddings) * (1.0 / math.sqrt(d))
    else:
        h = feats.reshape(-1, d)[sel].view(b, positions.size(1), d)
        logits = F.linear(h, model.tok_emb.weight)
    return logits


@torch.no_grad()
def eval_kv(model, n_samples=256 if not FAST_MODE else 32, seed=999):
    rng = random.Random(seed)
    model.eval()
    amp = BENCH_DEVICE.type == 'cuda'
    mc_correct = [0] * N_KEYS
    exact_correct = [0] * N_KEYS
    total = 0
    remaining = n_samples
    value_ids = torch.tensor(VAL_POOL, device=BENCH_DEVICE)
    while remaining > 0:
        b = min(16, remaining)
        remaining -= b
        seqs, answers = [], []
        for _ in range(b):
            s, a = make_kv_sample(rng)
            seqs.append(s)
            answers.append(a)
        x = torch.tensor(seqs, dtype=torch.long, device=BENCH_DEVICE)
        pos = torch.tensor([[p for p, _ in ans] for ans in answers], device=BENCH_DEVICE)
        with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
            logits = logits_at(model, x[:, :-1], pos)
        logits = logits.float()
        for row, ans in enumerate(answers):
            for qi, (p, value) in enumerate(ans):
                row_logits = logits[row, qi]
                exact_correct[qi] += int(int(row_logits.argmax().item()) == value)
                mc_scores = row_logits[value_ids]
                pred = int(value_ids[mc_scores.argmax()].item())
                mc_correct[qi] += int(pred == value)
                total += 1
    out = {
        'overall_mc': sum(mc_correct) / max(1, total),
        'overall_exact': sum(exact_correct) / max(1, n_samples * N_KEYS),
        'chance_mc': CHANCE,
        'n_samples': n_samples,
        'token_protocol': 'in-distribution (frecuencia real del train)',
    }
    for qi in range(N_KEYS):
        dist = QUERY_POSITIONS[qi] - (max(HEADER_SLOTS) + 2)
        out['mc_distance_%d' % dist] = mc_correct[qi] / max(1, n_samples)
        out['exact_distance_%d' % dist] = exact_correct[qi] / max(1, n_samples)
    return out


# ---------- 2) sonda de induccion (copia de secuencia repetida) ----------
IND_LEN = 8
IND_GAP = 48
IND_CHANCE = 1.0 / IND_LEN


def make_induction_sample(rng):
    core = rng.sample(order[100:400], IND_LEN)   # 8 tokens frecuentes distintos
    gap = [rng.choice(FILL_POOL) for _ in range(IND_GAP)]
    seq = [EOS_ID] + core + gap + [core[0]] + core[1:]
    answer_pos = 1 + IND_LEN + IND_GAP           # logits aqui predicen core[1]
    return seq, answer_pos, core[1], core


@torch.no_grad()
def eval_induction(model, n_samples=256 if not FAST_MODE else 32, seed=777):
    rng = random.Random(seed)
    model.eval()
    amp = BENCH_DEVICE.type == 'cuda'
    correct = 0
    remaining = n_samples
    while remaining > 0:
        b = min(16, remaining)
        remaining -= b
        seqs, apos, ans, cores = [], [], [], []
        for _ in range(b):
            s, p, a, c = make_induction_sample(rng)
            seqs.append(s); apos.append(p); ans.append(a); cores.append(c)
        x = torch.tensor(seqs, dtype=torch.long, device=BENCH_DEVICE)
        pos = torch.tensor([[p] for p in apos], device=BENCH_DEVICE)
        with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
            logits = logits_at(model, x[:, :-1], pos)
        logits = logits.float()[:, 0]
        for row in range(b):
            cands = torch.tensor(cores[row], device=BENCH_DEVICE)
            pred = int(cands[logits[row][cands].argmax()].item())
            correct += int(pred == ans[row])
    return {'overall': correct / max(1, n_samples), 'chance': IND_CHANCE,
            'protocol': 'induccion: 8 continuaciones posibles, gap 48'}


# ---------- 3) KV entrenado: fine-tune corto, presupuesto identico ----------
def finetune_kv(model, steps=300 if not FAST_MODE else 8, lr=1.5e-4, bs=16, seed=31337):
    rng = random.Random(seed)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.01)
    def sched(s):
        warm = min(1.0, (s + 1) / 20)
        return warm * 0.5 * (1 + math.cos(math.pi * min(1.0, s / steps)))
    amp = BENCH_DEVICE.type == 'cuda'
    for step in range(steps):
        for g in opt.param_groups:
            g['lr'] = lr * sched(step)
        seqs, answers = [], []
        for _ in range(bs):
            s, a = make_kv_sample(rng)
            seqs.append(s); answers.append(a)
        x = torch.tensor(seqs, dtype=torch.long, device=BENCH_DEVICE)
        y = torch.tensor([[v for _, v in ans] for ans in answers], device=BENCH_DEVICE)
        pos = torch.tensor([[p for p, _ in ans] for ans in answers], device=BENCH_DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
            logits = logits_at(model, x[:, :-1], pos)
            loss = F.cross_entropy(logits.float().reshape(-1, VOCAB_SIZE), y.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if step % 50 == 0:
            print('    ft step %d loss %.3f' % (step, float(loss.item())), flush=True)
    model.eval()
    return model


kv_report = {}
for arch, model in loaded.items():
    print('KV', arch, '...')
    zero = eval_kv(model)
    indu = eval_induction(model)
    print('  zero-shot in-dist  MC=%.1f%% (azar %.1f%%) | induccion=%.1f%% (azar %.1f%%)' % (
        100 * zero['overall_mc'], 100 * zero['chance_mc'],
        100 * indu['overall'], 100 * indu['chance']))
    model_bk = _copy.deepcopy(model)
    finetune_kv(model)
    trained = eval_kv(model, n_samples=256 if not FAST_MODE else 32)
    print('  KV entrenado       MC=%.1f%%  cerca=%.1f%%  lejos=%.1f%%' % (
        100 * trained['overall_mc'],
        100 * trained['mc_distance_24'], 100 * trained['mc_distance_176']))
    kv_report[arch] = {'zero_shot': zero, 'induction': indu, 'trained': trained}
    model.load_state_dict(model_bk.state_dict())  # no contaminar el resto de benches
    del model_bk

with open(os.path.join(SAVE_ROOT, 'kv_report.json'), 'w', encoding='utf-8') as f:
    json.dump(kv_report, f, indent=2)


## 11. Muestras de generación (cualitativo)


In [ ]:
PROMPTS = [
    'Once upon a time',
    'One day, a little girl named Anna',
    'Tom found a big red ball',
]


@torch.no_grad()
def generate_ids(model, prompt_ids, max_new=64, temperature=0.8, top_k=40):
    device = BENCH_DEVICE
    x = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    amp = device.type == 'cuda'
    kind = 'engrama' if hasattr(model, 'step_forward') else 'transformer'
    if kind == 'engrama':
        ids = model.generate(prompt_ids, max_new_tokens=max_new, temperature=temperature,
                             top_k=top_k, use_cache=True, eos_token_id=EOS_ID)
        return ids
    for _ in range(max_new):
        inp = x[:, -SEQ_LEN:]
        with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
            logits = model(inp)[:, -1].float()
        if temperature <= 0:
            nxt = int(logits.argmax(-1).item())
        else:
            logits = logits / temperature
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits = torch.where(logits < v[-1], torch.full_like(logits, -float('inf')), logits)
            probs = torch.softmax(logits, dim=-1)
            nxt = int(torch.multinomial(probs, 1).item())
        x = torch.cat([x, torch.tensor([[nxt]], device=device)], dim=1)
        if nxt == EOS_ID:
            break
    return x[0].tolist()


samples = {}
for arch, model in loaded.items():
    samples[arch] = {}
    print('\n====', arch, '====')
    for p in PROMPTS:
        ids = tokenizer.encode(p, add_bos=True, add_eos=False)
        try:
            out_ids = generate_ids(model, ids, max_new=48 if FAST_MODE else 80)
            text = tokenizer.decode(out_ids, skip_special_tokens=True)
        except Exception as exc:
            text = '[generacion fallida: %s]' % type(exc).__name__
        samples[arch][p] = text
        print('>', p)
        print(text[:500].replace('\n', ' '), '\n')

with open(os.path.join(SAVE_ROOT, 'samples.json'), 'w', encoding='utf-8') as f:
    json.dump(samples, f, indent=2, ensure_ascii=False)


## 12. Resumen completo para comparación


In [ ]:
def fmt_num(x, nd=4):
    if x is None:
        return '—'
    try:
        if isinstance(x, float) and (math.isnan(x) or math.isinf(x)):
            return '—'
    except Exception:
        return str(x)
    if isinstance(x, float):
        return ('%%.%df' % nd) % x
    return str(x)


def fmt_pct(x):
    if x is None:
        return '—'
    return '%.1f%%' % (100.0 * x)


rows = []
for arch in ARCHS:
    m = all_metrics.get(arch) or {}
    sc = scale_report.get(arch) or {}
    kv = kv_report.get(arch) or {}
    card = (m.get('card') if m else None) or cards.get(arch) or {}
    rows.append({
        'arch': arch,
        'titulo': card.get('title', arch),
        'params_M': (card.get('parameters') or 0) / 1e6,
        'gating': card.get('gating_mode', '—'),
        'trace_tap': card.get('trace_tap', '—'),
        'posicional': card.get('offset_mode') or card.get('positional') or '—',
        'atencion': 'no' if arch.startswith('engrama') else 'sí (SDPA causal)',
        'train_loss': m.get('final_train_loss'),
        'val_loss': m.get('best_val_loss'),
        'val_ppl': m.get('best_val_ppl'),
        'tok_s': m.get('tokens_per_sec_steady'),
        'sec_paso': m.get('sec_per_step'),
        'minutos': m.get('minutes'),
        'vram_train_GiB': m.get('peak_train_vram_gb'),
        'tokens_vistos': m.get('tokens_seen'),
        'skip_nan': m.get('skipped_nonfinite'),
        'forward_orden': sc.get('forward_order'),
        'forward_pendiente': sc.get('forward_slope'),
        'mem_orden': sc.get('memory_order'),
        'mem_pendiente': sc.get('memory_slope'),
        'decode_orden': sc.get('decode_order'),
        'decode_pendiente': sc.get('decode_slope'),
        'kv_zero': (kv.get('zero_shot') or {}).get('overall_mc'),
        'kv_zero_exact': (kv.get('zero_shot') or {}).get('overall_exact'),
        'kv_induction': (kv.get('induction') or {}).get('overall'),
        'kv_trained': (kv.get('trained') or {}).get('overall_mc'),
        'kv_d32': (kv.get('trained') or {}).get('mc_distance_24'),
        'kv_far': (kv.get('trained') or {}).get('mc_distance_176'),
    })

print('=' * 88)
print('RESUMEN COMPARATIVO  |  TinyStories 100M tok  |  GPT-2 vocab  |  seq 512  |  2xT4 DDP')
print('=' * 88)
hdr = ('%-22s %7s %10s %8s %10s %10s %8s %7s %11s %8s %8s %8s' % (
    'modelo', 'params', 'val_loss', 'ppl', 'tok/s', 's/paso', 'min', 'skip',
    'fwd orden', 'KV-zero', 'inducc', 'KV-ft'))
print(hdr)
print('-' * 88)
for r in rows:
    print('%-22s %6.2fM %10s %8s %10s %10s %8s %7s %11s %8s %8s %8s' % (
        r['arch'], r['params_M'],
        fmt_num(r['val_loss']), fmt_num(r['val_ppl'], 2),
        fmt_num(r['tok_s'], 0), fmt_num(r['sec_paso'], 3),
        fmt_num(r['minutos'], 1), fmt_num(r['skip_nan'], 0),
        r['forward_orden'] or '—',
        fmt_pct(r['kv_zero']), fmt_pct(r['kv_induction']), fmt_pct(r['kv_trained']),
    ))

print('\n--- Detalle arquitectónico ---')
print('%-22s %-16s %-10s %-22s %-20s' % ('modelo', 'gating', 'T0 tap', 'offsets/pos', 'atención'))
for r in rows:
    print('%-22s %-16s %-10s %-22s %-20s' % (
        r['arch'], str(r['gating']), str(r['trace_tap']), str(r['posicional']), r['atencion']))

print('\n--- Complejidad empírica (log-log, batch=1, AMP fp16) ---')
print('%-22s %-28s %-28s %-28s' % ('modelo', 'forward vs N', 'VRAM vs N', 'decode/token vs N'))
for r in rows:
    print('%-22s %-28s %-28s %-28s' % (
        r['arch'],
        '%s (b=%.2f)' % (r['forward_orden'] or '—', r['forward_pendiente'] or float('nan')),
        '%s (b=%.2f)' % (r['mem_orden'] or '—', r['mem_pendiente'] or float('nan')),
        '%s (b=%.2f)' % (r['decode_orden'] or '—', r['decode_pendiente'] or float('nan')),
    ))

print('\n--- VRAM forward (GiB) vs contexto ---')
print('%-22s' % 'modelo', end='')
for n in LENGTHS:
    print(' %8s' % ('N=%d' % n), end='')
print()
for arch in ARCHS:
    sc = scale_report.get(arch) or {}
    print('%-22s' % arch, end='')
    for peak in sc.get('peak_gb') or [None] * len(LENGTHS):
        print(' %8s' % (fmt_num(peak, 2) if peak is not None else '—'), end='')
    print()

print('\n--- Recuperación KV (MC/16 valores, azar = 6.2%%) ---')
print('%-22s %12s %12s %12s %10s %10s' % ('modelo', 'zero in-dist', 'inducción*', 'ft 250p', 'cerca', 'lejos'))
for r in rows:
    print('%-22s %12s %12s %12s %10s %10s' % (
        r['arch'], fmt_pct(r['kv_zero']), fmt_pct(r['kv_induction']),
        fmt_pct(r['kv_trained']), fmt_pct(r['kv_d32']), fmt_pct(r['kv_far'])))
print('  * inducción: azar = 12.5%%.  zero in-dist y ft: azar = 6.25%%.')

print('\n--- Entrenamiento ---')
print('%-22s %12s %12s %12s %10s %10s' % (
    'modelo', 'train_loss', 'val_loss', 'val_ppl', 'tok vistos', 'VRAM train'))
for r in rows:
    print('%-22s %12s %12s %12s %10s %10s' % (
        r['arch'], fmt_num(r['train_loss']), fmt_num(r['val_loss']), fmt_num(r['val_ppl'], 2),
        fmt_num(r['tokens_vistos'], 0), fmt_num(r['vram_train_GiB'], 2) + ' GiB'))

print('\nHiperparámetros compartidos:')
print('  dataset=TinyStories GPT-2  train_tokens=%s  valid_tokens=%s  seq=%d  epochs=%d' % (
    format(len(train_mm), ','), format(len(valid_mm), ','), SEQ_LEN, EPOCHS))
print('  nproc=%d  local_batch=%d  global_batch=%d  lr=%g  warmup=%d  cosine  clip=%s  AMP fp16' % (
    NPROC, LOCAL_BATCH, GLOBAL_BATCH, LR, WARMUP_STEPS, GRAD_CLIP))
print('  AdamW betas=(0.9, 0.95)  fused  CE lineal chunk=%d  compile=%s/%s' % (
    LINEAR_CHUNK, COMPILE, COMPILE_MODE))
print('  anti-NaN: CE fp32, GradScaler init_scale=2**12, GEMM fp16 reduction OFF, skip non-finite')

# Ranking helpers
def _key_loss(r):
    v = r['val_loss']
    return v if isinstance(v, (int, float)) and math.isfinite(v) else 1e9

ranked = sorted([r for r in rows if r['val_loss'] is not None], key=_key_loss)
print('\n--- Lectura rápida ---')
if ranked:
    print('  Mejor val_loss/PPL :', ranked[0]['arch'],
          'loss=%s ppl=%s' % (fmt_num(ranked[0]['val_loss']), fmt_num(ranked[0]['val_ppl'], 2)))
tok_ranked = sorted([r for r in rows if r['tok_s']], key=lambda r: r['tok_s'] or 0, reverse=True)
if tok_ranked:
    print('  Más tok/s train    :', tok_ranked[0]['arch'], fmt_num(tok_ranked[0]['tok_s'], 0))
kv_ranked = sorted([r for r in rows if r['kv_trained'] is not None], key=lambda r: r['kv_trained'] or 0, reverse=True)
if kv_ranked:
    print('  Mejor KV entrenado :', kv_ranked[0]['arch'], fmt_pct(kv_ranked[0]['kv_trained']))
kv0_ranked = sorted([r for r in rows if r['kv_zero'] is not None], key=lambda r: r['kv_zero'] or 0, reverse=True)
if kv0_ranked:
    print('  Mejor KV zero-shot :', kv0_ranked[0]['arch'], fmt_pct(kv0_ranked[0]['kv_zero']))
print('  Complejidad ENGRAMA (teoría): forward O(N), decode con caché ~O(1) por token (offsets fijos).')
print('  (La pendiente empírica a N<=512 se lee ~plana por overhead fijo; ver N hasta 2048.)')
print('  Complejidad Transformer (teoría): forward O(N²), decode con KV-cache O(N) por token.')
print('=' * 88)

summary = {
    'hyper': {
        'seq_len': SEQ_LEN, 'train_tokens': int(len(train_mm)), 'valid_tokens': int(len(valid_mm)),
        'epochs': EPOCHS, 'nproc': NPROC, 'local_batch': LOCAL_BATCH, 'global_batch': GLOBAL_BATCH,
        'lr': LR, 'warmup': WARMUP_STEPS, 'amp': True, 'compile': COMPILE,
    },
    'rows': rows,
    'metrics': all_metrics,
    'scale': scale_report,
    'kv': kv_report,
    'samples': samples,
}
with open(os.path.join(SAVE_ROOT, 'SUMMARY.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, default=str)
print('\nArtefactos en', SAVE_ROOT)
for p in sorted(Path(SAVE_ROOT).rglob('*')):
    if p.is_file() and p.stat().st_size < 50 * 1024 * 1024:
        print(' ', p.relative_to(SAVE_ROOT), '%.1f KB' % (p.stat().st_size / 1024))


## 13. Gráficas (loss, tok/s, VRAM, KV)

Si `matplotlib` no está, la celda 12 ya tiene la comparación completa en texto.


In [ ]:
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print('matplotlib no disponible:', exc)
else:
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    ax = axes[0, 0]
    for arch in ARCHS:
        hist = (all_metrics.get(arch) or {}).get('history') or []
        if hist:
            ax.plot([h['step'] for h in hist], [h['loss'] for h in hist], label=arch)
    ax.set_title('Train loss'); ax.set_xlabel('paso'); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    for arch in ARCHS:
        sc = scale_report.get(arch) or {}
        if sc.get('lengths'):
            ax.plot(sc['lengths'], sc['peak_gb'], marker='o', label=arch)
    ax.set_title('VRAM forward vs contexto'); ax.set_xlabel('N'); ax.set_ylabel('GiB')
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    for arch in ARCHS:
        sc = scale_report.get(arch) or {}
        if sc.get('lengths'):
            ax.loglog(sc['lengths'], sc['forward_sec'], marker='o', label=arch)
    ax.set_title('Forward time vs N (log-log)'); ax.set_xlabel('N'); ax.set_ylabel('s')
    ax.legend(); ax.grid(True, alpha=0.3, which='both')

    ax = axes[1, 1]
    labels, vals = [], []
    for arch in ARCHS:
        kv = kv_report.get(arch) or {}
        tr = (kv.get('trained') or {}).get('overall_mc')
        zs = (kv.get('zero_shot') or {}).get('overall_mc')
        if tr is not None:
            labels.append(arch.replace('engrama_', 'e_'))
            vals.append(100 * tr)
    if vals:
        xs = range(len(vals))
        ax.bar(xs, vals, label='entrenado 250p')
        zs_vals = [100 * ((kv_report.get(a) or {}).get('zero_shot') or {}).get('overall_mc', 0)
                   for a in ARCHS if ((kv_report.get(a) or {}).get('trained') or {}).get('overall_mc') is not None]
        ax.scatter(list(xs), zs_vals, color='tab:orange', zorder=3, label='zero-shot in-dist')
        ax.set_xticks(list(xs)); ax.set_xticklabels(labels)
        ax.axhline(100 * CHANCE, color='k', ls='--', label='azar 6.25%')
        ax.set_ylabel('% MC'); ax.set_title('KV retrieval (MC)'); ax.legend()
        ax.tick_params(axis='x', rotation=20)
    fig.tight_layout()
    fig_path = os.path.join(SAVE_ROOT, 'compare_plots.png')
    fig.savefig(fig_path, dpi=120)
    print('fig ->', fig_path)
    plt.show()
